### A5.1.12. Build Reference Counted Handle Class

**Problem:**

Build a handle class that tracks how many references share an underlying resource. The resource is destroyed only when the last reference is released.

**Explanation:**

Reference counting tracks shared ownership. A shared block holds the resource and a count. Each handle that points to the block increments the count. When a handle is released, it decrements the count. When the count reaches zero, the resource is destroyed.

How to build it:

1. Create a `RefBlock` that holds the resource and an integer count, starting at 1.
2. The `Handle` class stores a reference to the `RefBlock`.
3. `Handle.copy()`: create a new `Handle` pointing to the same `RefBlock` and increment the count.
4. `Handle.release()`: decrement the count. If it reaches zero, call a cleanup function on the resource.
5. The count is shared between all handles pointing to the same block.

**Example:**

Create a handle to a resource. Copy it twice. Release all three. The resource is cleaned up only on the last release.

In [ ]:
class RefBlock:
    def __init__(self, resource, cleanup):
        self.resource = resource
        self.cleanup = cleanup
        self.count = 1


class Handle:
    def __init__(self, resource, cleanup):
        self.block = RefBlock(resource, cleanup)

    def _from_block(self, block):
        new_handle = object.__new__(Handle)
        new_handle.block = block
        return new_handle

    def copy(self):
        self.block.count += 1
        return self._from_block(self.block)

    def release(self):
        self.block.count -= 1
        if self.block.count == 0:
            self.block.cleanup(self.block.resource)

    @property
    def resource(self):
        return self.block.resource

    @property
    def ref_count(self):
        return self.block.count


handle_a = Handle("DatabaseConnection", lambda resource: print(f"Destroyed: {resource}"))
print(f"After creation: ref_count={handle_a.ref_count}")

handle_b = handle_a.copy()
print(f"After copy: ref_count={handle_a.ref_count}")

handle_c = handle_b.copy()
print(f"After second copy: ref_count={handle_a.ref_count}")

handle_a.release()
print(f"After releasing A: ref_count={handle_b.ref_count}")

handle_b.release()
print(f"After releasing B: ref_count={handle_c.ref_count}")

handle_c.release()

**References:**

📘 Stroustrup, B. (2013). *The C++ Programming Language* — `shared_ptr` and Reference Counting

---

[⬅️ Previous: Build RAII File Descriptor Wrapper](./11_build_raii_file_descriptor_wrapper.ipynb) | [Next: Build PIMPL Wrapper Class ➡️](./13_build_pimpl_wrapper_class.ipynb)